In [19]:
import google.generativeai as genai
from dotenv import load_dotenv
# API KEY 정보 로드
load_dotenv()


# 내 키로 접근 가능한 모델 중 '임베딩' 기능이 있는 것만 출력
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(f"사용 가능한 모델명: {m.name}")

C:\workspace\class_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\junsu\AppData\Local\Temp\ipykernel_17556\1687564974.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


사용 가능한 모델명: models/gemini-embedding-001


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os


# API KEY 정보 로드
load_dotenv()


# LLM 엔진 (대답하는 기계)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 답변 한번에 나옴
# reponse = llm.invoke("안녕? 대한민국의 수도는 어디냐?")
# print(reponse.content)

# 스트리밍 출력
# for chunk in llm.stream("Java와 Python의 차이점을 짧게 알려줘."):
#     print(chunk.content, end="", flush=True)


# 1. 문서 로드 (PyMuPDFLoader 사용)
loader = PyMuPDFLoader("./files/profile_leejunsu.pdf") # 프로젝트 폴더에 있는 PDF 파일명
docs = loader.load()

# print(docs)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # 한 조각당 최대 글자 수
    chunk_overlap=50,   # 조각 간 겹치는 글자 수 (문맥 유실 방지용) chunk size에 10%~20%
    length_function=len,
    is_separator_regex=False
)

# 로드된 문서 분할
# loader.load()로 가져온 docs 객체 를 넣는다.
split_docs = text_splitter.split_documents(docs)

print(f"원본 문서 페이지수 : {len(docs)}")
print(f"분할된 조각(Chunk) 수 : {len(split_docs)}")


# 2. 임베딩 모델 설정 (텍스트를 숫자로 바꾸는 엔진)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


# 3. 벡터 DB 생성 (FAISS 사용)
# docs를 숫자로 바꿔서 메모리 DB(FAISS)에 저장합니다.
# vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore = FAISS.from_documents(docs, embeddings)

print(vectorstore)


# 4. 검색기(Retriever) 설정
retriever = vectorstore.as_retriever()

원본 문서 페이지수 : 3
분할된 조각(Chunk) 수 : 6


In [ ]:

# retriever = vectorstore.as_retriever()
# # 검색된 문서 조각들을 직접 확인해보기
# question = "문서에 지원자 이름이 뭐야?"
# relevant_docs = retriever.invoke(question)
#
# print(f"--- 검색된 문서 조각 개수: {len(relevant_docs)} ---")
# for i, doc in enumerate(relevant_docs):
#     print(f"[{i}] 내용: {doc.page_content[:100]}...")
#     print(f"[{i}] 출처: {doc.metadata}\n")

In [35]:

# 5. 프롬프트 템플릿 설정 (질문 형식을 정의)
template = """다음 문맥을 활용하여 질문에 답하세요:
{context}

질문: {question}
답변:"""
prompt = PromptTemplate.from_template(template)

# 6. 체인(Chain) 생성 (LCEL 방식: 부품들을 파이프로 연결)
chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
)

# 7. 실행
# ans = chain.invoke("이 문서에서 가장 중요한 내용은 뭐야?")
# print(ans)
for ans in chain.stream("문서에 지원자의 이름과 경력 회사명을 알려줘"):
    print(ans, end="", flush=True)

지원자의 이름은 이준수입니다.

경력 회사명은 다음과 같습니다:
*   주식회사 유알피
*   디지털존
*   케이비즈소프트(KBIZ-SOFT)
*   아이티앤